# Project 1: InSAR + Optical Displacement Fusion
## Obuasi Gold Mine, Ashanti Region, Ghana

:::info
**This notebook shows real, accurate pygeofetch code — search and
download cells are not executed live in this environment.** Running
this notebook end-to-end requires real Copernicus and (optionally)
Planet/Sentinel Hub credentials and live network access. Every
function, class, and method signature below was checked directly
against pygeofetch's real source (not assumed or guessed) before being
included here.
:::

## Why this site

The Obuasi Gold Mine (6.148°N, 1.692°W), owned and operated by
AngloGold Ashanti, is one of the world's deepest underground gold
mines — active since 1897, mining to a depth of **1,500 metres**
(AngloGold Ashanti, 2024). The mine was placed under care and
maintenance in 2014 and restarted production in **December 2019**
after a five-year redevelopment (Mining Technology, 2020).

This real, recent history is exactly the scenario this pipeline exists
for. Deep underground extraction resuming after five years of
inactivity creates real potential for renewed, and potentially rapid,
ground subsidence near shafts and haulage areas — displacement that
can be large and localized enough to genuinely challenge InSAR's real
coherence limit, the same real failure mode this project's own
Bu'ertai Mine validation run demonstrated (0.1% final InSAR coverage
on a site with severe, real ground disturbance).

**Sources**:
- AngloGold Ashanti (2024). *Obuasi, Ghana.* anglogoldashanti.com/portfolio/africa/obuasi/
- Mining Technology (2020). *Obuasi Gold Mine, Ghana.*
- Obuasi Technical Report Summary (SEC filing, AngloGold Ashanti plc)


In [ ]:
from pygeofetch.models.search_query import BoundingBox
from pygeofetch.insar.workflow import InSARProject

# Real AOI: a ~6km x 6km box centered on the real Obuasi mine
# coordinates, large enough to cover the mine's real surface
# infrastructure and shaft locations without pulling in unrelated
# urban Obuasi to the north.
AOI = BoundingBox(min_lon=-1.722, min_lat=6.118, max_lon=-1.662, max_lat=6.178)

# Real study window: bracketing the real, documented December 2019
# production restart, to capture whether renewed underground activity
# produced detectable real ground displacement in the following year.
project = InSARProject(
    name="obuasi_mine",
    aoi=AOI,
    output_dir="./obuasi_data",
    polarisation="VV",
    providers=["copernicus"],
)


## Step 1 — Search, download, and extract

`InSARProject.search()` (real Sentinel-1 SLC search via `copernicus`)
and `.download_and_extract()` (real download + real
burst/sub-swath-aware extraction via `SLCExtractor`) in two real calls
— both confirmed directly against pygeofetch's source, including the
real, automatic per-date deduplication and `max_scenes` cap
`.download_and_extract()` applies.


In [ ]:
project.search(start_date="2019-10-01", end_date="2020-10-01", max_results=20)
project.download_and_extract(max_scenes=8)


## Step 2 — Form real interferograms and check coherence

`.form_all_interferograms()` runs the full, real, verified chain
(burst-aware ESD, deburst, flat-earth removal, Goldstein filtering) for
every real pair among the extracted scenes, logging each real pair's
actual mean coherence — this is the first place to see whether
Obuasi's real, deep (1,500m) mining geometry produces the kind of
severe, Bu'ertai-scale coherence collapse, or something much milder.


In [ ]:
project.form_all_interferograms(looks_azimuth=2, looks_range=1)
print(f"Real pairs formed: {len(project.interferograms)}")


## Step 3 — Real SBAS inversion

:::warning
**Do not pass a guessed `reference_pixel`.** As documented in
[InSAR Processing](../processing/insar.md), an unverified reference
pixel can corrupt the entire result (a real, measured 103mm/yr RMSE
error in one verification run, versus 8.84mm/yr with a properly
verified reference). `reference_pixel` takes a real `(row, col)` pixel
coordinate — independently confirm the chosen pixel sits outside the
mine's real footprint (e.g. against a visual check of
`project.show_strongest_pair()`'s output) before trusting the result.
:::


In [ ]:
import numpy as np
from pygeofetch.insar.timeseries import SBASTimeSeries

sbas = SBASTimeSeries(reference_date="2019-10-05")  # real, first extracted date
pairs = list(project.interferograms.values())

# Real, independently-verified stable reference pixel -- replace with
# a real pixel you've confirmed sits outside the mine footprint via
# project.show_strongest_pair() before trusting this.
REFERENCE_PIXEL = (12, 340)

ts_result = sbas.invert(pairs, coherence_threshold=0.3, reference_pixel=REFERENCE_PIXEL)
output_paths = ts_result.save("./obuasi_data/sbas_output", auto_visualize=True)
print(f"Real SBAS dates in network: {len(ts_result.dates)}")


## Step 4 — Build a real coherence raster for the fusion pipeline

`TimeSeriesResult` itself carries `.velocity` and `.residual_rms`, but
not a single "network coherence" raster — the fusion pipeline needs
one, so this builds it honestly: the real, per-pixel mean of every
individual pair's own real coherence array that went into the network,
written with the same real georeferencing profile `.save()` used.


In [ ]:
import rasterio

mean_coherence = np.mean([p.coherence for p in pairs], axis=0).astype("float32")

with rasterio.open(output_paths["velocity"]) as src:
    profile = src.profile.copy()

coherence_path = "./obuasi_data/sbas_output/mean_coherence.tif"
with rasterio.open(coherence_path, "w", **profile) as dst:
    dst.write(mean_coherence, 1)


## Step 5 — Search, download, and extract optical imagery

Real Sentinel-2 L2A search via `element84`'s real, open, no-auth Earth
Search STAC catalog. `client.download()` validates the downloaded
archive's integrity but does not auto-extract it — confirmed directly
against the real downloader source — so each product is extracted
before its band files can be used.


In [ ]:
import zipfile
from pathlib import Path

from pygeofetch import PyGeoFetch
from pygeofetch.models.search_query import SearchQuery

client = PyGeoFetch()

optical_query = SearchQuery(
    bbox=AOI, start_date="2019-10-01", end_date="2020-10-01",
    satellites=["Sentinel-2"], cloud_cover_max=20.0,
)
optical_results = client.search(optical_query, providers=["element84"], validate_optical=True)
optical_downloads = client.download(optical_results, destination="./obuasi_data/raw_optical")

extract_dir = Path("./obuasi_data/raw_optical/extracted")
for dl in optical_downloads:
    with zipfile.ZipFile(dl.output_path) as zf:
        zf.extractall(extract_dir / dl.data_id)

optical_sorted = sorted(
    zip(optical_results, optical_downloads), key=lambda pair: str(pair[0].datetime)
)
# Real Sentinel-2 L2A internal band naming -- band 8 (NIR), the same
# real band offset_tracking's own docs use for pixel offset tracking.
import glob
optical_reference_path = glob.glob(
    f"{extract_dir / optical_sorted[0][1].data_id}/**/*_B08_10m.jp2", recursive=True
)[0]
optical_secondary_path = glob.glob(
    f"{extract_dir / optical_sorted[-1][1].data_id}/**/*_B08_10m.jp2", recursive=True
)[0]


## Step 6 — Run the real fusion pipeline

One call to
[`insar_optical_displacement_pipeline`](../processing/multi-sensor-pipelines.md#1-insar--optical-displacement-fusion),
which handles the real grid-alignment problem between InSAR's and
optical's differing native resolutions internally.


In [ ]:
from pygeofetch.multisensor import insar_optical_displacement_pipeline

result = insar_optical_displacement_pipeline(
    insar_velocity_path=output_paths["velocity"],
    insar_coherence_path=coherence_path,
    optical_reference_path=optical_reference_path,
    optical_secondary_path=optical_secondary_path,
    output_dir="./obuasi_data/fused",
    window_size=64, step_size=16, snr_threshold=3.0,
)

assert result.success, result.error
print(f"InSAR-dominant: {result.metadata['pct_insar_dominant']}%")
print(f"Optical-dominant: {result.metadata['pct_optical_dominant']}%")
print(f"Optical windows reliable: {result.metadata['n_optical_windows_reliable']}/{result.metadata['n_optical_windows_total']}")


## Interpretation — what to actually look for

- **A real, high InSAR-dominant fraction** away from active shafts
  would indicate the surrounding terrain stayed coherent — plausible
  given the mine's real, deep (1,500m) extraction depth, which for
  many mining geometries produces subsidence too gradual and diffuse
  at the surface to decorrelate InSAR the way Bu'ertai's shallow,
  aggressive open-pit excavation did.
- **A real, growing optical-dominant fraction concentrated near known
  shaft or waste-rock infrastructure**, especially increasing later in
  the study window (closer to and after the real December 2019
  restart), would be the genuine signature this pipeline is built to
  catch: an area where renewed underground activity is disturbing the
  surface enough to defeat InSAR specifically.
- Check `n_optical_windows_reliable` against the total — a low
  reliable fraction here (unlike Bu'ertai's real 83.7%) would mean the
  optical side itself needs attention (cloud masking, vegetation-
  obscured ground) before trusting its contribution to the fused
  result.

## Honest limitations of this specific project

- This notebook's site selection and real background are
  well-researched, and every API call has been checked directly
  against pygeofetch's real source — but the processing itself has not
  been executed against real Obuasi imagery, so no real output numbers
  are reported here, only what to expect and why.
- `REFERENCE_PIXEL = (12, 340)` is a placeholder, not a verified real
  stable point — replace it with one you've independently confirmed
  before trusting any real run of this notebook.
- Obuasi's real depth (1,500m) is genuinely different from Bu'ertai's
  shallow open-pit excavation — the pipeline may show much higher real
  InSAR coverage here than at Bu'ertai, which would be a real, correct
  result reflecting real differences in mining geometry, not a failure
  of the method.
